# R04 · Embedded Linux boot, device tree and the character-device model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/05_realtime_linux/R04_embedded_linux_boot_device_tree_and_the_characte.ipynb)

**Track:** Real-time and Linux · **Skill axis (sheet):** Real-time and Linux · **Time:** about 60 min

**Skill Hub concepts:** Linux boot chain and device tree; Character device driver; Wait queues and kernel locking contexts

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("R04")

## Map: from power-on to `/dev/imu0`

```
 Boot ROM ──> SPL / TF-A (DRAM init, secure world) ──> U-Boot (loads kernel + DTB) ──> Linux kernel
     ──> parses the device tree, probes drivers whose `compatible` matches ──> init (systemd) ──> your app
```

**Device tree:** a description of hardware the kernel cannot discover (I2C, SPI, GPIO lines, interrupts).

| Property | Meaning |
|---|---|
| `compatible = "vendor,part"` | which driver binds to this node |
| `reg` | address: register block for SoC devices, bus address for I2C, chip select for SPI |
| `#address-cells` / `#size-cells` | how many 32-bit cells `reg` uses in child nodes (I2C/SPI children: 1 and 0) |
| `interrupts = <GIC_SPI 37 IRQ_TYPE_LEVEL_HIGH>` | interrupt specifier; GIC shared peripheral interrupt n has ID 32 + n |
| `status = "okay"` / `"disabled"` | enable or disable the node |
| `spi-max-frequency`, `spi-cpha`, `spi-cpol` | SPI clock and mode (lab D03) |

**Character device driver** (`struct file_operations`): `open`, `read`, `write`, `poll`, `unlocked_ioctl`, `release`;
registered under a major/minor number (`dev_t = major << 20 | minor`) and appears as `/dev/name` via udev.
A blocking `read` with no data sleeps on a **wait queue** (`wait_event_interruptible`) until the interrupt path calls
`wake_up`; with `O_NONBLOCK` it returns `-EAGAIN` instead.

| Lock | May sleep while holding? | Usable from interrupt context? | Typical use |
|---|---|---|---|
| `spinlock_t` (`spin_lock_irqsave`) | **no** | yes | data shared with an interrupt handler, held for a few instructions |
| `struct mutex` | yes | **no** | longer sections in process context, may call `copy_to_user` (which can fault and sleep) |

In [ ]:
from roboprep import clab
clab.ensure_apt("dtc", "device-tree-compiler")

## Task 1: order the boot chain

In [ ]:
stages = ["init (systemd)", "U-Boot", "Linux kernel + DTB", "Boot ROM", "SPL / TF-A"]
boot_order = None  # TODO: list the stages in order
lab.check("boot", boot_order)

## Task 2: describe two sensors in a device tree
Under the I2C controller add `imu@68` (`compatible = "invensense,icm42688"`, `reg = <0x68>`, `interrupts = <37 4>`).
Under the SPI controller add `encoder@0` for the AS5047P (`compatible = "ams,as5047p"`, chip select 0,
10 MHz, SPI mode 1, which means the `spi-cpha` flag).

In [ ]:
%%writefile board.dts
/dts-v1/;
/ {
    compatible = "rdl,robot-board";
    #address-cells = <1>;
    #size-cells = <1>;

    i2c@40005400 {
        compatible = "st,stm32f7-i2c";
        reg = <0x40005400 0x400>;
        #address-cells = <1>;
        #size-cells = <0>;
        status = "okay";
        /* TODO: imu@68 */
    };

    spi@40013000 {
        compatible = "st,stm32h7-spi";
        reg = <0x40013000 0x400>;
        #address-cells = <1>;
        #size-cells = <0>;
        status = "okay";
        /* TODO: encoder@0 */
    };
};

In [ ]:
print(clab.sh("dtc -I dts -O dtb -o board.dtb board.dts && dtc -I dtb -O dts board.dtb | sed -n '1,40p'").stdout)
lab.check("dts")

## Task 3: a character device's read path, modeled in Python
Same semantics as a driver: a 64-byte buffer filled by "interrupts" (`write`), a blocking `read` that sleeps on a
condition variable (the wait queue) until data arrives, and `nonblock=True` raising `BlockingIOError` (EAGAIN).

In [ ]:
import threading

class CharDev:
    def __init__(self, size=64):
        self.buf, self.size = bytearray(), size
        self.lock = threading.Lock()
        self.wq = threading.Condition(self.lock)       # the wait queue

    def write(self, data: bytes) -> int:
        """Producer side (think: interrupt handler). Returns bytes accepted (drops what does not fit)."""
        raise NotImplementedError  # TODO

    def read(self, n: int, nonblock=False, timeout=None) -> bytes:
        raise NotImplementedError  # TODO

lab.check("chardev", CharDev)

In [ ]:
%%writefile r04_chardev.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/r04.h
// Grade it with lab.check("chardev", "r04_chardev.c"). For C++, name the file .cpp in both places.
#include "r04.h"

struct CharDev {
    size_t size;  /* TODO: buffer, length, a mutex and a condition variable */
};

CharDev *chardev_create(size_t size) { (void)size; return NULL; /* TODO */ }
void chardev_destroy(CharDev *d) { (void)d; /* TODO */ }
size_t chardev_write(CharDev *d, const uint8_t *data, size_t len) { (void)d; (void)data; (void)len; return 0; /* TODO */ }
long chardev_read(CharDev *d, uint8_t *out, size_t n, bool nonblock, double timeout_s) {
    (void)d; (void)out; (void)n; (void)nonblock; (void)timeout_s;
    return 0; /* TODO: -11 when nonblock and empty; otherwise wait on the condition until data or timeout */
}

In [ ]:
# lab.check("chardev", "r04_chardev.c")  # uncomment after writing the C/C++ cell above

In [ ]:
locks = {
    "data shared between an interrupt handler and a syscall": None,
    "section that calls copy_to_user()": None,
    "config list touched only by process-context code that may sleep": None,
    "a few register writes inside a hard interrupt handler": None,
}
lab.check("locks", locks)

## Checkpoint: Math
**M1 (Apply).** `dev_t` value for major 240, minor 3 (`major << 20 | minor`)?

**M2 (Apply).** The `reg` value of `imu@68`, in decimal?

**M3 (Apply).** `interrupts = <GIC_SPI 37 ...>`. The GIC interrupt ID?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why does Linux on Arm need a device tree when a PC does not?
- a) Arm kernels are smaller
- b) on-chip and board-level buses like I2C and SPI cannot be enumerated, so the hardware must be described; PCs discover devices through ACPI and PCI enumeration
- c) U-Boot requires it

**K2 (Analyze).** Walk through what happens from `cat /dev/imu0` with an empty buffer until data comes back, naming the
wait queue, the interrupt path and the lock used where.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `r04.h`).
3. In `roboprep/checks/r04.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()